In [ ]:
PLOT_HIST=True

In [ ]:
from pathlib import Path
from typing import Dict, Optional, Tuple
import warnings
import zipfile
import xml.etree.ElementTree as ET

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import math
from shapely.geometry import LineString, Polygon, box

In [ ]:
VAR_NAMES = {'hur': 'Relative Humidity (%)',
 'hurs': 'Near-Surface Relative Humidity (%)',
 'hus': 'Specific Humidity (1)',
 'huss': 'Near-Surface Specific Humidity (1)',
 'pr': 'Precipitation (kg m-2 s-1)',
 'ps': 'Surface Air Pressure (Pa)',
 'psl': 'Sea Level Pressure (Pa)',
 'sfcWind': 'Near-Surface Wind Speed (m s-1)',
 'ta': 'Air Temperature (K)',
 'tas': 'Near-Surface Air Temperature (K)',
 'tasmax': 'Daily Maximum Near-Surface Air Temperature (K)',
 'tasmin': 'Daily Minimum Near-Surface Air Temperature (K)',
 'ts': 'Surface Temperature (K)'}


In [ ]:
def parse_cmip6_filename(filename: str) -> Optional[Dict[str, str]]:
    stem = Path(filename).stem
    parts = stem.split("_")
    if len(parts) < 7:
        return None
    return {
        "variable": parts[0],
        # "table": parts[1],
        "model": parts[2],
        # "experiment": "_".join(parts[3:-3]),
        # "ensemble": parts[-3],
        # "grid": parts[-2],
        "date_range": parts[-1],
    }


def load_inventory(base_path: str) -> pd.DataFrame:
    root = Path(base_path)
    if not root.exists():
        return pd.DataFrame()

    records = []
    for scenario_dir in sorted([d for d in root.iterdir() if d.is_dir()]):
        for nc_file in sorted(scenario_dir.glob("*.nc")):
            meta = parse_cmip6_filename(nc_file.name)
            if meta:
                meta["scenario"] = scenario_dir.name
                meta["filepath"] = str(nc_file)
                records.append(meta)

    return pd.DataFrame(records)

data_path =  "cmip6_brazil/monthly"
df_inventory = load_inventory(data_path)


In [ ]:
def format_temporal_range_label(date_range: str) -> str:
    parts = str(date_range).split("-")
    if len(parts) == 2 and len(parts[0]) >= 4 and len(parts[1]) >= 4:
        return f"{parts[0][:4]}-{parts[1][:4]}"
    return str(date_range)

var_df = df_inventory[df_inventory["variable"] == 'pr'].copy()

var_df.loc[:, "date_range"] = var_df.loc[:, "date_range"]\
                                            .apply(format_temporal_range_label)


In [ ]:
print("Number of unique scenario-date_range combinations:", len(var_df[['scenario','date_range']].astype(str).apply(tuple, axis=1).unique()))

In [ ]:
from pathlib import Path
import zipfile
import xml.etree.ElementTree as ET

import geopandas as gpd
from shapely.geometry import LineString


def parse_kmz_with_attributes(kmz_file: str) -> gpd.GeoDataFrame:
    path = Path(kmz_file)
    if not path.exists():
        raise FileNotFoundError(f"KMZ file not found: {path}")

    ns = {"kml": "http://www.opengis.net/kml/2.2"}
    records = []

    with zipfile.ZipFile(path, "r") as zf:
        with zf.open("doc.kml") as kml:
            tree = ET.parse(kml)
            root = tree.getroot()

            for pm in root.findall(".//kml:Placemark", ns):
                name = pm.findtext("kml:name", default="", namespaces=ns)

                # ExtendedData (important!)
                ext_data = {}
                for data in pm.findall(".//kml:ExtendedData//kml:Data", ns):
                    key = data.attrib.get("name")
                    value = data.findtext("kml:value", default="", namespaces=ns)
                    if key:
                        ext_data[key] = value

                coords_node = pm.find(".//kml:LineString/kml:coordinates", ns)
                if coords_node is None or not coords_node.text:
                    continue

                coords = []
                for token in coords_node.text.strip().split():
                    parts = token.split(",")
                    if len(parts) >= 2:
                        lon, lat = float(parts[0]), float(parts[1])
                        coords.append((lon, lat))

                if len(coords) >= 2:
                    geom = LineString(coords)

                    record = {
                        "name": name,
                        **ext_data,
                        "geometry": geom,
                    }
                    records.append(record)

    if not records:
        raise ValueError("No valid Placemarks found.")

    return gpd.GeoDataFrame(records, crs="EPSG:4326")


# usage
gdf = parse_kmz_with_attributes("ISA ENERGIA BRASIL.kmz")

In [ ]:
import geopandas as gpd
import numpy as np
from sklearn.cluster import DBSCAN


def cluster_lines_distance(gdf, eps_km=100):
    gdf = gdf.copy()

    # centroids
    centroids = gdf.geometry.centroid

    # (lat, lon) for haversine
    coords = np.array([(pt.y, pt.x) for pt in centroids])
    coords_rad = np.radians(coords)

    # DBSCAN with haversine
    db = DBSCAN(
        eps=eps_km / 6371.0,  # km → radians
        min_samples=1,
        metric="haversine"
    ).fit(coords_rad)

    gdf["cluster"] = db.labels_
    return gdf

gdf_clustered = cluster_lines_distance(gdf, eps_km=50)

In [ ]:
from cartopy.io import shapereader
import cartopy.crs as ccrs
def load_brazil_geometry():

    shp = shapereader.natural_earth(resolution="10m", category="cultural", name="admin_0_countries")
    reader = shapereader.Reader(shp)
    return [rec.geometry for rec in reader.records() if rec.attributes.get("ADMIN") == "Brazil"]

def load_brazil_states():
    shp = shapereader.natural_earth(
        resolution="10m",
        category="cultural",
        name="admin_1_states_provinces"
    )
    reader = shapereader.Reader(shp)

    states = []
    for rec in reader.records():
        if rec.attributes.get("admin") == "Brazil":
            states.append({
                "geometry": rec.geometry,
                "name": rec.attributes.get("name"),
                "abbr": rec.attributes.get("postal")  # sometimes available
            })
    return states

states = load_brazil_states()
brazil = load_brazil_geometry()


In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib.colors as mcolors

clusters = sorted(gdf_clustered["cluster"].unique())
n = len(clusters)

# pick exactly n colors
colors = plt.cm.nipy_spectral(np.linspace(0, 0.9, n))

cmap = mcolors.ListedColormap(colors)
color_map = {c: colors[i] for i, c in enumerate(clusters)}
# cmap = plt.get_cmap("nipy_spectral", (n+1))  # +1 to avoid 0=white

# map cluster → color
color_map = {c: cmap(i+1) for i, c in enumerate(clusters)}

colors = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#8c564b",  # brown
    "#e377c2",  # pink
    "#7f7f7f",  # gray
    "#bcbd22",  # yellow-green
    "#17becf",  # cyan
    "#000000",  # black
    "#ffd700",  # gold
]
subplot_kw = {"projection": ccrs.PlateCarree()}
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=subplot_kw)

# plot each cluster with its exact color
for i, c in enumerate(clusters):

    subset = gdf_clustered[gdf_clustered["cluster"] == c]
    subset.plot(
        ax=ax,
        color=colors[i],
        linewidth=2,
        label=f"área {c}"
    )

# Brazil boundary
ax.add_geometries(
    brazil,
    crs=ccrs.PlateCarree(),
    facecolor="none",
    edgecolor="black",
    linewidth=1.0,
    zorder=5,
)

for state in states:
    ax.add_geometries(
        [state["geometry"]],
        crs=ccrs.PlateCarree(),
        facecolor="none",
        edgecolor="gray",
        linewidth=0.5,
        zorder=2,
    )

    # --- label (centroid) ---
    centroid = state["geometry"].centroid
    ax.text(
        centroid.x,
        centroid.y,
        state["abbr"] if state["abbr"] else state["name"],
        fontsize=6,
        ha="center",
        transform=ccrs.PlateCarree(),
        zorder=4,
    )

# ax.coastlines(resolution="10m", linewidth=0.5)
ax.set_extent([-75, -33, -35, 7], crs=ccrs.PlateCarree())

ax.set_title(f"Áreas de Estudos \n{n} áreas agrupadas por proximidade geográfica")
ax.set_axis_off()

# legend with only used clusters
ax.legend(title="Áreas", loc="lower left")
plt.savefig("ead_figures/areas_estudo.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# import matplotlib.pyplot as plt

# for empresa, group in gdf_clustered.groupby("cluster"):
#     fig, ax = plt.subplots(figsize=(8, 8))

#     group.plot(column="EMPRESA", cmap="tab10", ax=ax, linewidth=1)

#     ax.set_title(f"Grupo de linhas {empresa} | {len(group['EMPRESA'].unique())} empresas")
#     ax.set_axis_off()

#     plt.show()

In [ ]:
import os 

def infer_lat_lon_names(ds: xr.Dataset) -> Tuple[str, str]:
    lat_candidates = ("lat", "latitude", "y")
    lon_candidates = ("lon", "longitude", "x")
    lat_name = next((n for n in lat_candidates if n in ds.coords or n in ds.dims), None)
    lon_name = next((n for n in lon_candidates if n in ds.coords or n in ds.dims), None)
    if lat_name is None or lon_name is None:
        raise ValueError("Latitude/longitude coordinates were not found.")
    return lat_name, lon_name


def subset_da_to_bbox(da: xr.DataArray, lat_name: str, lon_name: str, bounds: Tuple[float, float, float, float]) -> xr.DataArray:
    """Subset DataArray to bounding box. bounds = [lon_min, lat_min, lon_max, lat_max]"""
    lon_vals = da[lon_name]
    if float(lon_vals.max()) > 180:
        da = da.assign_coords({lon_name: (((lon_vals + 180) % 360) - 180)}).sortby(lon_name)

    lon_min, lat_min, lon_max, lat_max = bounds
    lon_min = math.floor(lon_min)
    lat_min = math.floor(lat_min)
    lon_max = math.ceil(lon_max)
    lat_max = math.ceil(lat_max)

    lat_vals = np.asarray(da[lat_name].values, dtype=float)
    lon_vals = np.asarray(da[lon_name].values, dtype=float)

    lat_slice = slice(lat_min, lat_max) if lat_vals[0] < lat_vals[-1] else slice(lat_max, lat_min)
    lon_slice = slice(lon_min, lon_max) if lon_vals[0] < lon_vals[-1] else slice(lon_max, lon_min)

    return da.sel({lat_name: lat_slice, lon_name: lon_slice})


def pad_bounds_km(bounds: Tuple[float, float, float, float], pad_km: float = 10.0) -> Tuple[float, float, float, float]:
    lon_min, lat_min, lon_max, lat_max = [float(v) for v in bounds]
    lat_mid = (lat_min + lat_max) / 2.0

    pad_lat = pad_km / 111.0
    cos_lat = max(np.cos(np.deg2rad(lat_mid)), 1e-6)
    pad_lon = pad_km / (111.32 * cos_lat)

    return (
        lon_min - pad_lon,
        lat_min - pad_lat,
        lon_max + pad_lon,
        lat_max + pad_lat,
    )




In [ ]:
if PLOT_HIST:
    
    # Iterate through each EMPRESA/cluster group
    for cluster, group in gdf_clustered.groupby("cluster"):

        empresas = ", ".join(group['EMPRESA'].unique())
        n_linhas = len(group['LINHA'].unique())

        raw_bounds = group.total_bounds  # [lon_min, lat_min, lon_max, lat_max]
        bounds = pad_bounds_km(raw_bounds, pad_km=10.0)

        print(f"\n{'='*60}")
        print(f"Empresas: {empresas}")
        print(f"Nº de Linhas: {n_linhas}")
        print(f"Bounds original: {raw_bounds}")
        print(f"Bounds com padding (~10 km): {bounds}")
        print(f"{'='*60}")
        
        # Group var_df by scenario and date_range
        for (scenario, date_range), scenario_group in var_df.groupby(['scenario', 'date_range']):
            n_models = len(scenario_group)
            ncols = min(3, n_models)
            nrows = math.ceil(n_models / ncols)
            
            fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
            axes = np.atleast_1d(axes).reshape(-1)
            
            ax_idx = 0
            for _, row in scenario_group.iterrows():
                try:
                    with xr.open_dataset(row["filepath"]) as ds:
                        if row["variable"] not in ds.data_vars:
                            continue
                        
                        da = ds[row["variable"]]
                        lat_name, lon_name = infer_lat_lon_names(ds)
                        
                        seconds_in_month = da['time'].dt.days_in_month * 24 * 60 * 60

                        # Subset to cluster bounds
                        da_subset = subset_da_to_bbox(da, lat_name, lon_name, bounds)
                        da_subset = da_subset * seconds_in_month # convert from kg m-2 s-1 to m/month

                        values = da_subset.values.flatten() 
                        values = values[np.isfinite(values)]
                        
                        if len(values) == 0:
                            print(f"  {row['model']}: No valid data in bbox")
                            continue

                        # Clean values (important!)
                        values_clean = values[~np.isnan(values)]

                        # Plot histogram
                        ax = axes[ax_idx]
                        ax.hist(values_clean, bins=30, edgecolor='black', alpha=0.7, color='lightgray')

                        # --- Stats ---
                        q5, q10, q50, q90, q95 = np.percentile(values_clean, [5, 10, 50, 90, 95])
                        # vmin, vmax = np.min(values_clean), np.max(values_clean)

                        # --- Lines ---
                        ax.axvline(q50, linestyle='-', linewidth=2, label=f"Median: {q50:.1f}")
                        ax.axvline(q90, linestyle='-.', linewidth=1.5, label=f"Q90: {q90:.1f}")
                        ax.axvline(q95, linestyle=':',  linewidth=2.0, label=f"Q95: {q95:.1f}")

                        # Labels
                        ax.set_title(f"{row['model']}")
                        ax.set_xlabel("Precipitação mm/mês")
                        # ax.set_xlabel("Velocidade do Vento (m s-1)")
                        ax.set_ylabel("Frequência")

                        ax.legend(fontsize=8)

                        ax_idx += 1
                        
                except Exception as e:
                    print(f"  Error processing {row['model']}: {str(e)}")
                    continue
            
            # Hide unused subplots
            for idx in range(ax_idx, len(axes)):
                axes[idx].axis('off')
            
            scenario = "Histórica" if scenario == "historical" else scenario.replace("_", "-", 1).replace("_", ".", 1)
            # title = f"{empresa} | Cluster {cluster} | {scenario.upper()} {date_range}\n{VAR_NAMES.get(var_df.iloc[0]['variable'], var_df.iloc[0]['variable']).split}"
            title = f"Precipitação (mm/mês) {scenario} {date_range} | Área de estudo: {cluster}\nEmpresas: {empresas} | {n_linhas} linhas"
            fig.suptitle(title, fontsize=12, fontweight='bold')
            plt.tight_layout()

            os.makedirs(f"ead_figures/cluster_{cluster}", exist_ok=True)       
            plt.savefig(f"ead_figures/cluster_{cluster}/precipitacao_{scenario}_{date_range}.png", dpi=300) 

            # plt.show()        

In [ ]:
# MIN overall temperature map per dataset, restricted to grid cells intersecting group lines

import os
from shapely.geometry import box


def coordinate_edges(coord_values: np.ndarray) -> np.ndarray:
    values = np.asarray(coord_values, dtype=float)
    if values.size == 1:
        delta = 0.5
        return np.array([values[0] - delta, values[0] + delta], dtype=float)

    diffs = np.diff(values)
    edges = np.empty(values.size + 1, dtype=float)
    edges[1:-1] = (values[:-1] + values[1:]) / 2.0
    edges[0] = values[0] - diffs[0] / 2.0
    edges[-1] = values[-1] + diffs[-1] / 2.0
    return edges


def build_line_intersection_mask(lat_centers: np.ndarray, lon_centers: np.ndarray, line_union) -> np.ndarray:
    lat_edges = coordinate_edges(lat_centers)
    lon_edges = coordinate_edges(lon_centers)

    cell_geometries, cell_i_lat, cell_i_lon = [], [], []
    for i in range(len(lat_centers)):
        y0, y1 = sorted((float(lat_edges[i]), float(lat_edges[i + 1])))
        for j in range(len(lon_centers)):
            x0, x1 = sorted((float(lon_edges[j]), float(lon_edges[j + 1])))
            cell_geometries.append(box(x0, y0, x1, y1))
            cell_i_lat.append(i)
            cell_i_lon.append(j)

    cells = gpd.GeoDataFrame(
        {"i_lat": cell_i_lat, "i_lon": cell_i_lon},
        geometry=cell_geometries,
        crs="EPSG:4326",
    )
    selected = cells.loc[cells.intersects(line_union)]
    # selected = cells.loc[cells.intersects(line_union.buffer(0.05))]

    mask = np.zeros((len(lat_centers), len(lon_centers)), dtype=bool)
    if not selected.empty:
        mask[selected["i_lat"].to_numpy(), selected["i_lon"].to_numpy()] = True
    return mask


def build_min_intersection_panel(filepath: str, variable_name: str, bounds, line_union):
    with xr.open_dataset(filepath) as ds:
        if variable_name not in ds.data_vars:
            return None

        da = ds[variable_name]
        lat_name, lon_name = infer_lat_lon_names(ds)


        da = subset_da_to_bbox(da, lat_name, lon_name, bounds)
        seconds_in_month = da['time'].dt.days_in_month * 24 * 60 * 60
        da = da * seconds_in_month

        if da.sizes.get(lat_name, 0) == 0 or da.sizes.get(lon_name, 0) == 0:
            return None

        if "time" in da.dims:
            da = da.min(dim="time", skipna=True)

        # for dim in list(da.dims):
        #     if dim not in (lat_name, lon_name):
        #         da = da.isel({dim: 0})
        # da = da.squeeze()

        # lat = np.asarray(da[lat_name].values, dtype=float)
        # lon = np.asarray(da[lon_name].values, dtype=float)

        for dim in list(da.dims):
            if dim not in (lat_name, lon_name):
                da = da.isel({dim: 0})
        
        # Removido: da.squeeze() 

        lat = np.atleast_1d(da[lat_name].values).astype(float)
        lon = np.atleast_1d(da[lon_name].values).astype(float)

        if lat.size == 0 or lon.size == 0:
            return None

        mask = build_line_intersection_mask(lat, lon, line_union)
        if not np.any(mask):
            return None
        
        mask_da = xr.DataArray(mask, coords={lat_name: da[lat_name], lon_name: da[lon_name]}, dims=(lat_name, lon_name))
        da_masked = da.where(mask_da)

        units = da.attrs.get("units", "N/A")
        # Força a matriz a ter 2 dimensões (lat, lon)
        values = np.asarray(da_masked.values, dtype=float).reshape(len(lat), len(lon))

    return {
        "lat": lat,
        "lon": lon,
        "values": values,
        "units": units,
    }

    #     mask_da = xr.DataArray(mask, coords={lat_name: da[lat_name], lon_name: da[lon_name]}, dims=(lat_name, lon_name))
    #     da_masked = da.where(mask_da)

    #     units = da.attrs.get("units", "N/A")
    #     values = np.asarray(da_masked.values, dtype=float)

    # return {
    #     "lat": lat,
    #     "lon": lon,
    #     "values": values,
    #     "units": units,
    # }





# Same division logic as histograms: cluster -> scenario/date_range -> model subplots
for cluster, group in gdf_clustered.groupby("cluster"):

    empresas = ", ".join(group["EMPRESA"].unique())
    n_linhas = len(group["LINHA"].unique()) if "LINHA" in group.columns else len(group)

    raw_bounds = group.total_bounds
    bound_map = pad_bounds_km(raw_bounds, pad_km=10.0)
    bounds = pad_bounds_km(raw_bounds, pad_km=50.0)

    line_union = group.geometry.union_all()

    print(f"\n{'='*80}")
    print(f"Cluster: {cluster} | Empresas: {empresas} | Linhas: {n_linhas}")
    print(f"Bounds: {bounds}")
    print(f"{'='*80}")

    for (scenario, date_range), scenario_group in var_df.groupby(["scenario", "date_range"]):
        n_models = len(scenario_group)
        ncols = min(3, n_models)
        nrows = math.ceil(n_models / ncols)

        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows), 
            constrained_layout=True
        )

        axes = np.atleast_1d(axes).reshape(-1)

        panels = []
        vmins, vmaxs = [], []

        for _, row in scenario_group.iterrows():
            try:
                panel = build_min_intersection_panel(
                    filepath=row["filepath"],
                    variable_name='pr',
                    bounds=bounds,
                    line_union=line_union,
                )
                if panel is None:
                    continue

                panel["model"] = row["model"]
                panels.append(panel)

                vals = panel["values"]
                if np.isfinite(vals).any():
                    vmins.append(float(np.nanmin(vals)))
                    vmaxs.append(float(np.nanmax(vals)))
            except Exception as exc:
                print(f"  Error in {row['model']} ({scenario} {date_range}): {exc}")
                continue

        if not panels:
            plt.close(fig)
            print(f"  No valid panels for {scenario} {date_range}")
            continue

        vmin = min(vmins) if vmins else None
        vmax = max(vmaxs) if vmaxs else None

        # mesh = None
        # for ax, panel in zip(axes, panels):
        #     mesh = ax.pcolormesh(
        #         panel["lon"],
        #         panel["lat"],
        #         panel["values"],
        #         shading="auto",
        #         cmap="coolwarm",
        #         vmin=vmin,
        #         vmax=vmax,
        #     )

        mesh = None
        for ax, panel in zip(axes, panels):
            mesh = ax.pcolormesh(
                coordinate_edges(panel["lon"]), # Usa as bordas de longitude
                coordinate_edges(panel["lat"]), # Usa as bordas de latitude
                panel["values"],
                shading="flat",                 # Troque "auto" por "flat"
                cmap="coolwarm",
                vmin=vmin,
                vmax=vmax,
            )

            group.plot(ax=ax, color="black", linewidth=1.0, alpha=0.9)
            ax.set_xlim(bound_map[0], bound_map[2])
            ax.set_ylim(bound_map[1], bound_map[3])
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
            ax.set_title(panel["model"], fontsize=9)

        for ax in axes[len(panels):]:
            ax.axis("off")

        scenario_label = "Histórica" if scenario == "historical" else scenario.replace("_", "-", 1).replace("_", ".", 1)
        # title = f"Velocidade do Vento Média Mensal (m s-1) {scenario} {date_range} | Área de estudo: {cluster}\nEmpresas: {empresas} | {n_linhas} linhas"
        title = f"Precipitação Mínima (mm/mês) {scenario_label} {date_range} | Área de estudo: {cluster}\nEmpresas: {empresas} | {n_linhas} linhas"
        fig.suptitle(
            title,
            fontsize=12,
            fontweight="bold",
        )

        if mesh is not None:
            units = panels[0].get("units", "N/A")
            cbar = fig.colorbar(mesh, ax=axes[:len(panels)], shrink=0.85)
            # cbar.set_label(f"Velocidade do Vento (m s-1)")
            cbar.set_label(f"Precipitação (mm/mês)")

        # plt.tight_layout()

        os.makedirs(f"ead_figures/cluster_{cluster}", exist_ok=True)
        plt.savefig(
            f"ead_figures/cluster_{cluster}/min_prep_intersection_{scenario}_{date_range}.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()
